<a href="https://colab.research.google.com/github/janaeeeeeee/data1001-Labs-/blob/main/Lab_3_Data_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

# Load the dataset
df = pd.read_csv("Messy_Employee_dataset.csv")

# looks at dimension
print(df.shape)

# this looks at the first 5 rows of the dataset
print(df.head())

# this looks at the column names, dtat types and non-null counts
df.info()

# summary statistics for numeric columns
print(df.describe())

# Count of missing values per column
print("Missing Count:")
print(df.isna().sum())

# Percentage of missing values per column
print("Missing Percentage:")
print(df.isna().mean() * 100)

(1020, 12)
  Employee_ID First_Name Last_Name   Age   Department_Region    Status  \
0     EMP1000        Bob     Davis  25.0   DevOps-California    Active   
1     EMP1001        Bob     Brown   NaN       Finance-Texas    Active   
2     EMP1002      Alice     Jones   NaN        Admin-Nevada   Pending   
3     EMP1003        Eva     Davis  25.0        Admin-Nevada  Inactive   
4     EMP1004      Frank  Williams  25.0  Cloud Tech-Florida    Active   

    Join_Date     Salary                       Email       Phone  \
0    4/2/2021   59767.65       bob.davis@example.com -1651623197   
1   7/10/2020   65304.66       bob.brown@example.com -1898471390   
2   12/7/2023   88145.90     alice.jones@example.com -5596363211   
3  11/27/2021   69450.99       eva.davis@example.com -3476490784   
4    1/5/2022  109324.61  frank.williams@example.com -1586734256   

  Performance_Score  Remote_Work  
0           Average         True  
1         Excellent         True  
2              Good         Tr

In [10]:
# 1. Check for whole-row duplicates
print("Whole row exact duplicates:")
print(df.duplicated().sum())

# 2. Check for key column duplicates (ID and Email)
print("Duplicate Employee_IDs:")
print(df['Employee_ID'].duplicated().sum())

print("Duplicate Emails:")
print(df['Email'].duplicated().sum())

# Checks row count before removing duplicates
rows_before = len(df)

# Drop exact duplicate rows
df = df.drop_duplicates()

# Checks row count after
rows_after = len(df)

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Duplicates removed: {rows_before - rows_after}")

# Handles missing values by filling with the median
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Salary'] = df['Salary'].fillna(df['Salary'].median())

df['Join_Date'] = pd.to_datetime(df['Join_Date'])

# 2. Convert negative phone values to positive
df['Phone'] = df['Phone'].abs()

# B4. CLEAN TEXT / SPLIT COLUMNS
# Split 'Department_Region' into two separate columns: 'Department' and 'Region'
df[['Department', 'Region']] = df['Department_Region'].str.split('-', expand=True)

# Remove extra spacing around text
df['Department'] = df['Department'].str.strip()
df['Region'] = df['Region'].str.strip()





Whole row exact duplicates:
0
Duplicate Employee_IDs:
0
Duplicate Emails:
956
Rows before: 1020
Rows after:  1020
Duplicates removed: 0


In [11]:

# C1. DETECT OUTLIERS (IQR METHOD)


# 1. Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = df['Salary'].quantile(0.25)
Q3 = df['Salary'].quantile(0.75)

# 2. Calculate IQR
IQR = Q3 - Q1

# 3. Calculate lower and upper bounds
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

# 4. Flag any rows outside these bounds
outliers = df[(df['Salary'] < lower_bound) | (df['Salary'] > upper_bound)]

print(f"Q1 (25th Percentile): ${Q1:,.2f}")
print(f"Q3 (75th Percentile): ${Q3:,.2f}")
print(f"IQR: ${IQR:,.2f}")
print(f"Lower Bound: ${lower_bound:,.2f}")
print(f"Upper Bound: ${upper_bound:,.2f}")
print(f"Number of Outliers Found: {len(outliers)}")

Q1 (25th Percentile): $68,811.23
Q3 (75th Percentile): $100,372.66
IQR: $31,561.43
Lower Bound: $21,469.09
Upper Bound: $147,714.81
Number of Outliers Found: 0


In [16]:
#C2

# C2. MERGE TWO TABLES

# 1. Create a lookup table for regions
region_lookup = pd.DataFrame({
    'Region': ['California', 'Texas', 'Nevada', 'Florida', 'New York', 'Illinois'],
    'State_Code': ['CA', 'TX', 'NV', 'FL', 'NY', 'IL'],
    'State_Tax_Rate': [0.133, 0.000, 0.000, 0.000, 0.109, 0.0495]
})

# 2. Merge the lookup table with our main DataFrame on 'Region'
merged_df = pd.merge(df, region_lookup, on='Region', how='left')

# Preview the result
print(merged_df[['Employee_ID', 'First_Name', 'Last_Name', 'Region', 'State_Code', 'State_Tax_Rate']].head(3))

  Employee_ID First_Name Last_Name      Region State_Code  State_Tax_Rate
0     EMP1000        Bob     Davis  California         CA           0.133
1     EMP1001        Bob     Brown       Texas         TX           0.000
2     EMP1002      Alice     Jones      Nevada         NV           0.000


In [15]:

# C3. RESHAPE DATA (WIDE TO LONG)

# 1. Take a small 3-row subset in wide format
wide_subset = merged_df[['Employee_ID', 'Department', 'Region', 'Salary']].head(3)

print("BEFORE: Wide Format ")
print(wide_subset)

# 2. Reshape from Wide to Long using melt()
long_df = pd.melt(
    wide_subset,
    id_vars=['Employee_ID', 'Department'],
    value_vars=['Region', 'Salary'],
    var_name='Attribute',
    value_name='Value'
)

print(" AFTER: Long (Tidy) Format ")
print(long_df)

BEFORE: Wide Format 
  Employee_ID Department      Region    Salary
0     EMP1000     DevOps  California  59767.65
1     EMP1001    Finance       Texas  65304.66
2     EMP1002      Admin      Nevada  88145.90
 AFTER: Long (Tidy) Format 
  Employee_ID Department Attribute       Value
0     EMP1000     DevOps    Region  California
1     EMP1001    Finance    Region       Texas
2     EMP1002      Admin    Region      Nevada
3     EMP1000     DevOps    Salary    59767.65
4     EMP1001    Finance    Salary    65304.66
5     EMP1002      Admin    Salary     88145.9


Janae’ Hill 500002975
https://www.kaggle.com/datasets/desolution01/messy-employee-dataset
In Part A, I loaded the Messy_Employee_dataset.csv file, which contains 1,020 rows and 12 columns. The dataset includes numeric columns (Age, Salary, Phone) and text columns like Employee_ID, First_Name, Last_Name, Department_Region, Status, Join_Date, Email, Performance_Score, and Remote_Work. Missing data showed up in Age (211 missing, 20.69%) and Salary (24 missing, 2.35%), which are both Missing at Random (MAR) since there is no systematic reason for them being left out. While there were no exact whole-row duplicates or duplicate Employee_IDs, Email had 956 repeated values due to placeholder text.
In Part B, I ran drop_duplicates(), keeping all 1,020 rows. For missing values in Age and Salary, I use their medians so I wouldn't lose data or mess up the overall distributions. I fixed data types by converting Join_Date to datetime and taking the absolute value of Phone to clear out negative numbers. For text cleaning, I split Department_Region into separate Department and Region columns using .str.split('-') and trimmed extra spaces with .str.strip().
In Part C, I checked for outliers in Salary using the IQR method (Q1: $68,811.23, Q3: $100,372.66, IQR: $31,561.43). The lower and upper bounds came out to [$21,469.09, $147,714.81], which gave 0 outliers since all salary figures were within reasonable bounds. I then joined the main dataset with a 6-row regional lookup table containing state codes and tax rates using a left join (how="left") on Region so no employee records were dropped. Finally, I took a small subset of the merged data and unpivoted it from wide to long format using pd.melt(), showing that wide format works best for looking at individual records while long format is better for statistical analysis and plotting.
